# Using Machine Learning to Classify Land Cover Types

**Goal:** produce annual land cover classifications for Oregon and Washington (2017–2025).

**Reference labels:** [GLanCE](https://www.nature.com/articles/s41597-023-02798-5) — a global time series
land cover training/validation dataset — filtered to its North America records and Level-1 classes
(Water, Ice/Snow, Developed, Barren/Sparse, Trees, Shrubs, Herbaceous). Each GLanCE record is a segment
of years over which a location's land cover class is constant, so it is exploded into one row per year
before modeling.

**Predictors:** Google/DeepMind's [AlphaEarth Satellite Embedding](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL)
dataset — a 64-dimensional learned embedding per 10 m pixel per year (2017 onward), pulled from Earth
Engine at each GLanCE point/year. Because this embedding is itself the output of a pretrained geospatial
foundation model, classification only requires a lightweight model on top of it.

**Approach:** several classifiers are trained on the embeddings and compared on a shared, spatially
grouped held-out test set (grouped by GLanCE segment so no location leaks between train and test):
- **Random Forest** and **XGBoost**, both plain and hyperparameter-tuned via grouped cross-validation
- **XGBoost (spatial)** — the same tuned XGBoost with per-band neighborhood mean/std features added, to
  give the tree model some spatial context
- **Shallow MLP** — a small neural network on the 64-dim embeddings, plain and with early stopping
- **CNN** — a small convolutional network on a k×k neighborhood of embeddings around each point (rather
  than a single pixel), plain and with early stopping

The notebook ends by comparing every model's accuracy, per-class F1, and training time, both overall and
restricted to the Washington/Oregon subset of the test set.

## Setup


In [ ]:
%load_ext autoreload
%autoreload 2

# ── Load libraries and functions ─────────────────────────────────────────────────────
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import shape
import os
import ee
import geemap
import time
import joblib

from Glance_Class_Definitions import class1_dict, class2_dict
from Embedding_Utils import get_embeddings, EMBEDDING_BANDS, get_patch_arrays, align_patch_arrays, neighborhood_stats
from Tree_Ensemble import split_data, fit_random_forest, fit_xgboost, tune_random_forest, tune_xgboost, rf_validation_curve, _LabelDecodingClassifier
from MLP import fit_mlp, _MLPClassifier
from CNN import fit_cnn, _CNNClassifier
from Constants import DATA_DIR, MODEL_DIR
from Evaluation import plot_feature_importances, evaluate_model, compare_models, plot_training_curve, plot_validation_curve

# ── Initiate GEE Project ─────────────────────────────────────────────────────────────
ee.Initialize(project='turnkey-lacing-391919')

# ── Define Variables ─────────────────────────────────────────────────────────────────
start_year = 2017
download_embeddings = True #If True, sample any missing years' embeddings from GEE; if False, only load years already cached locally
refit_models = True # When set to True, this will cause the models to be refit. Available saved models will be loaded when False

## Data Preparation

Load GLanCE, filter to North America / 2017+, explode segments to one row per year, join AlphaEarth embeddings, and make the grouped train/test split.

In [ ]:
# ── Read in Glance Data ──────────────────────────────────────────────────────────────

geojson_path = os.path.join('Data', 'bu_glance_training_dataV1.geojson')
#https://www.nature.com/articles/s41597-023-02798-5#Sec10
#https://source.coop/boston-university/bu-glance

# Large file (~1.1 GB / ~1.87M features) — pyogrio is the fast, memory-efficient reader.
glance = gpd.read_file(geojson_path, engine='pyogrio')


In [ ]:
# ── Filter Glance Data ───────────────────────────────────────────────────────────────
glance_NA = glance.loc[glance['Continent_Code']==1,] # Filter to just North America

glance_NA_filtered = glance_NA.loc[glance_NA['End_Year']>=start_year,]
print(glance_NA_filtered['Glance_Class_ID_level1'].isna().sum()==0) #Make sure that there are No NAs present

glance_NA_filtered = glance_NA_filtered[['Lat','Lon','Glance_ID','Start_Year','End_Year','Glance_Class_ID_level1','Glance_Class_ID_level2','geometry']]

end_year = glance_NA_filtered['End_Year'].unique().max()

In [ ]:
# ── Per Year Explosion ───────────────────────────────────────────────────────────────
# One row per year each segment is active, clipped to AlphaEarth's 2017+ range.
expanded_Glance = glance_NA_filtered.copy()

# ── Explode segments into per-year rows ──────────────────────────────────────────────
expanded_Glance['Year'] = expanded_Glance.apply(
    lambda r: list(range(max(int(r['Start_Year']), start_year), int(r['End_Year']) + 1)),
    axis=1
)

expanded_Glance = expanded_Glance.drop(columns=['Start_Year', 'End_Year'])

expanded_Glance = expanded_Glance.explode('Year', ignore_index=True)

# ── Split into Level-1 (all rows) and Level-2 (drop 'Unknown' = 0) tables ────────────
expanded_Glance_Class1 = expanded_Glance.drop(columns = ['Glance_Class_ID_level2'])

expanded_Glance_Class2 = expanded_Glance.loc[expanded_Glance['Glance_Class_ID_level2'] != 0,]
expanded_Glance_Class2 = expanded_Glance_Class2.drop(columns = ['Glance_Class_ID_level1'])

# ── QC checks ────────────────────────────────────────────────────────────────────────
print([0] not in expanded_Glance_Class2['Glance_Class_ID_level2'].unique())
print(expanded_Glance_Class1.shape[0] > expanded_Glance_Class2.shape[0])
print(expanded_Glance_Class1['Year'].min(), expanded_Glance_Class1['Year'].max())


In [ ]:
# ── Read in Embeddings and Organize ──────────────────────────────────────────────────
embeddings_dir = os.path.join('Data', 'Embeddings')

# ── Drop embedding columns from a prior run so this cell is safe to re-run ───────────
expanded_Glance_Class1 = expanded_Glance_Class1.drop(columns=EMBEDDING_BANDS, errors='ignore')
expanded_Glance_Class2 = expanded_Glance_Class2.drop(columns=EMBEDDING_BANDS, errors='ignore')

# ── Sample/cache once against Class1 (the superset), then join into both ─────────────
unique_year_points = expanded_Glance_Class1[['Glance_ID', 'Year', 'Lat', 'Lon']].drop_duplicates()
embeddings_df = get_embeddings(unique_year_points, cache_dir=embeddings_dir, download_embeddings=download_embeddings)
embeddings_df['Year'] = embeddings_df['Year'].astype(expanded_Glance_Class1['Year'].dtype)
expanded_Glance_Class1 = expanded_Glance_Class1.merge(embeddings_df, on=['Glance_ID', 'Year'], how='inner')
expanded_Glance_Class2 = expanded_Glance_Class2.merge(embeddings_df, on=['Glance_ID', 'Year'], how='inner')

# ── QC checks ────────────────────────────────────────────────────────────────────────
print(f'Class1 rows with embeddings: {expanded_Glance_Class1.shape[0]} / {unique_year_points.shape[0]}')
print(f'Class2 rows with embeddings: {expanded_Glance_Class2.shape[0]}')
print(f'{expanded_Glance_Class1[EMBEDDING_BANDS].isna().sum().sum()} missing embedding values')


In [ ]:
# ── Split data into training and test ────────────────────────────────────────────────
expanded_Glance_Class1_Clean = expanded_Glance_Class1.drop(columns = ["geometry","Year","Lat","Lon"])
                                                 
x_train, x_test, y_train, y_test = split_data(expanded_Glance_Class1_Clean, "Glance_Class_ID_level1")
print(x_train.head())


## Models

Each cell fits (or loads, when `refit_models=False`) one variant, evaluates it on the shared held-out test set, and saves diagnostics to `Model_Outputs/`.

### Random Forest & XGBoost

In [ ]:
# ── Random Forest ────────────────────────────────────────────────────────────────────
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = ""
filename = f'random_forest_model{filename_ext}.joblib'

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_random_forest(x_train, y_train)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'random_forest_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'random_forest_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ─────────────────────────────────────────────────────────────────────────
print(f'Training time: {training_time_sec:.2f} seconds')
plot_feature_importances(
    mdl, x_train, title='Random Forest — feature importances',
    save_path=os.path.join(MODEL_DIR, f'random_forest_importances{filename_ext}.png'))
rf_metrics = evaluate_model(
    mdl, x_test, y_test, class_map=class1_dict,
    title='Random Forest — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'random_forest_confusion{filename_ext}.png'))

# ── Overfitting diagnostic (RF has no epochs -> validation curve instead) ────────────
if refit_models:
    groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]
    vc_param, vc_train, vc_val = rf_validation_curve(
        x_train, y_train, groups_train, param_name='min_samples_leaf',
        param_range=(1, 2, 4, 8, 16))
    plot_validation_curve(
        vc_param, vc_train, vc_val, param_name='min_samples_leaf', scoring='macro F1',
        title='Random Forest — validation curve',
        save_path=os.path.join(MODEL_DIR, f'random_forest_valcurve{filename_ext}.png'))


In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────────────────────────
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = ""
filename = f'XGBoost_model{filename_ext}.joblib'

# ── Glance_ID groups (grouped val split for the boosting-round loss curve) ───────────
groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_xgboost(x_train, y_train, groups=groups_train)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'XGBoost_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'XGBoost_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ─────────────────────────────────────────────────────────────────────────
print(f'Training time: {training_time_sec:.2f} seconds')
plot_feature_importances(
    mdl, x_train, title='XGBoost — feature importances',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_importances{filename_ext}.png'))
xgb_metrics = evaluate_model(
    mdl, x_test, y_test, class_map=class1_dict,
    title='XGBoost — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_confusion{filename_ext}.png'))

# ── Overfitting diagnostic (train vs val mlogloss per boosting round) ────────────────
plot_training_curve(
    getattr(mdl, 'history_', None), title='XGBoost — loss curve',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_losscurve{filename_ext}.png'))


In [ ]:
# ── Random Forest (grouped-CV tuned) ─────────────────────────────────────────────────
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = "_groupedCV"
filename = f'random_forest_tuned_model{filename_ext}.joblib'

# ── Glance_ID groups (so no segment spans CV folds) ──────────────────────────────────
groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec, best_params = tune_random_forest(x_train, y_train, groups_train)
    print(f'Best params: {best_params}')
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'random_forest_tuned_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
        f.write(f'best_params: {best_params}\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'random_forest_tuned_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ─────────────────────────────────────────────────────────────────────────
print(f'Search time: {training_time_sec:.2f} seconds')
plot_feature_importances(
    mdl, x_train, title='Random Forest (tuned) — feature importances',
    save_path=os.path.join(MODEL_DIR, f'random_forest_tuned_importances{filename_ext}.png'))
rf_tuned_metrics = evaluate_model(
    mdl, x_test, y_test, class_map=class1_dict,
    title='Random Forest (tuned) — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'random_forest_tuned_confusion{filename_ext}.png'))


In [ ]:
# ── XGBoost (grouped-CV tuned) ───────────────────────────────────────────────────────
# Searches depth/learning_rate/subsample/gamma/reg_alpha/max_delta_step/...; n_estimators
# is not tuned -- the winner is refit with a high cap + early stopping, so the number of
# boosting rounds is chosen by validation mlogloss.
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = "_groupedCV"
filename = f'XGBoost_tuned_model{filename_ext}.joblib'

# ── Glance_ID groups (so no segment spans CV folds) ──────────────────────────────────
groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec, best_params = tune_xgboost(x_train, y_train, groups_train)
    print(f'Best params: {best_params}')
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'XGBoost_tuned_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
        f.write(f'best_params: {best_params}\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'XGBoost_tuned_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ─────────────────────────────────────────────────────────────────────────
print(f'Search time: {training_time_sec:.2f} seconds')
best_iter = getattr(mdl, 'best_iteration', None)
if best_iter is not None:
    print(f'Early stopping chose best_iteration={best_iter} (of up to 2000 rounds)')
plot_feature_importances(
    mdl, x_train, title='XGBoost (tuned) — feature importances',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_tuned_importances{filename_ext}.png'))
xgb_tuned_metrics = evaluate_model(
    mdl, x_test, y_test, class_map=class1_dict,
    title='XGBoost (tuned) — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_tuned_confusion{filename_ext}.png'))

# ── Overfitting diagnostic (train vs val mlogloss; early stopping trims rounds) ──────
plot_training_curve(
    getattr(mdl, 'history_', None), title='XGBoost (tuned) — loss curve',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_tuned_losscurve{filename_ext}.png'))


### XGBoost with spatial features

Tuned XGBoost on the 64 center-pixel embeddings plus per-band neighborhood mean/std (192 features).

In [ ]:
# ── XGBoost (spatial) ────────────────────────────────────────────────────────────────
# Tuned XGBoost on the 64 center-pixel embeddings + per-band neighborhood mean/std
# (192 features). Stats come from the cached CNN patches (no extra GEE download).
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = "_spatial"
filename = f'XGBoost_nbhd_tuned_model{filename_ext}.joblib'

# ── Neighborhood mean/std features (cached to Data/, reloaded if present) ────────────
nbhd_path = os.path.join('Data', f'nbhd_features_r{patch_radius}.parquet')
if os.path.exists(nbhd_path):
    nbhd_features = pd.read_parquet(nbhd_path)
    print(f'Loaded cached nbhd_features from {nbhd_path}: {nbhd_features.shape}')
if not os.path.exists(nbhd_path) or len(nbhd_features) != len(expanded_Glance_Class1):
    nbhd_features = neighborhood_stats(X_patch_all)
    nbhd_features.to_parquet(nbhd_path)
    print(f'Saved nbhd_features to {nbhd_path}: {nbhd_features.shape}')

# ── Build the 192-feature tables on the existing train/test split ────────────────────
nbhd_features.index = expanded_Glance_Class1.index
x_train_nbhd = pd.concat([x_train, nbhd_features.loc[x_train.index]], axis=1)
x_test_nbhd = pd.concat([x_test, nbhd_features.loc[x_test.index]], axis=1)
groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec, best_params = tune_xgboost(x_train_nbhd, y_train, groups_train)
    print(f'Best params: {best_params}')
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'XGBoost_nbhd_tuned_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
        f.write(f'best_params: {best_params}\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'XGBoost_nbhd_tuned_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ─────────────────────────────────────────────────────────────────────────
print(f'Search time: {training_time_sec:.2f} seconds')
best_iter = getattr(mdl, 'best_iteration', None)
if best_iter is not None:
    print(f'Early stopping chose best_iteration={best_iter} (of up to 2000 rounds)')
plot_feature_importances(
    mdl, x_train_nbhd, title='XGBoost (spatial) — feature importances',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_nbhd_tuned_importances{filename_ext}.png'))
xgb_nbhd_tuned_metrics = evaluate_model(
    mdl, x_test_nbhd, y_test, class_map=class1_dict,
    title='XGBoost (spatial) — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_nbhd_tuned_confusion{filename_ext}.png'))

# ── Overfitting diagnostic (train vs val mlogloss per boosting round) ────────────────
plot_training_curve(
    getattr(mdl, 'history_', None), title='XGBoost (spatial) — loss curve',
    save_path=os.path.join(MODEL_DIR, f'XGBoost_nbhd_tuned_losscurve{filename_ext}.png'))


### Shallow MLP

In [ ]:
# ── Shallow MLP ──────────────────────────────────────────────────────────────────────
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = ""
filename = f'MLP_model{filename_ext}.joblib'

# ── Glance_ID groups (grouped val split for the loss curve) ──────────────────────────
groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_mlp(x_train, y_train, groups=groups_train, verbose=True, epochs=100)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'MLP_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'MLP_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate (no feature importances for an MLP) ─────────────────────────────────────
print(f'Training time: {training_time_sec:.2f} seconds')
mlp_metrics = evaluate_model(
    mdl, x_test, y_test, class_map=class1_dict,
    title='Shallow MLP — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'MLP_confusion{filename_ext}.png'))

# ── Overfitting diagnostic (train vs val loss per epoch) ─────────────────────────────
plot_training_curve(
    getattr(mdl, 'history_', None), title='Shallow MLP — loss curve',
    save_path=os.path.join(MODEL_DIR, f'MLP_losscurve{filename_ext}.png'))


In [ ]:
# ── Shallow MLP (early stopping) ─────────────────────────────────────────────────────
# Stops when val loss plateaus and restores the best-epoch weights (epochs is the cap).
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = "_tuned"
filename = f'MLP_model{filename_ext}.joblib'

# ── Glance_ID groups (grouped val split to early-stop on) ────────────────────────────
groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_mlp(x_train, y_train, groups=groups_train,
                                     early_stopping_rounds=15, epochs=300, verbose=True)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'MLP_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'MLP_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ─────────────────────────────────────────────────────────────────────────
print(f'Training time: {training_time_sec:.2f} seconds')
best_epoch = getattr(mdl, 'best_epoch_', None)
if best_epoch is not None:
    print(f'Early stopping restored best epoch {best_epoch + 1}')
mlp_tuned_metrics = evaluate_model(
    mdl, x_test, y_test, class_map=class1_dict,
    title='Shallow MLP (tuned) — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'MLP_confusion{filename_ext}.png'))

# ── Overfitting diagnostic (train vs val loss per epoch; stops near best epoch) ──────
plot_training_curve(
    getattr(mdl, 'history_', None), title='Shallow MLP (tuned) — loss curve',
    save_path=os.path.join(MODEL_DIR, f'MLP_losscurve{filename_ext}.png'))


### CNN (embedding patches)

Uses the k×k×64 neighborhood window around each point instead of the single center pixel.

In [ ]:
# ── Patch Embeddings (for CNN) ───────────────────────────────────────────────────────
# Raw k x k x 64 window per point (radius=2 -> 5x5). The aligned array X_patch_all is
# cached to Data/ and reloaded if present, so sampling + alignment isn't repeated.
patch_dir = os.path.join('Data', 'Embeddings_Patch')
patch_radius = 2
patch_all_path = os.path.join('Data', f'X_patch_all_r{patch_radius}.npy')

# ── Load cache, or sample + align + save ─────────────────────────────────────────────
if os.path.exists(patch_all_path):
    X_patch_all = np.load(patch_all_path)
    print(f'Loaded cached X_patch_all from {patch_all_path}: {X_patch_all.shape}')
if not os.path.exists(patch_all_path) or X_patch_all.shape[0] != len(expanded_Glance_Class1):
    patch_year_points = expanded_Glance_Class1[['Glance_ID', 'Year', 'Lat', 'Lon']].drop_duplicates()
    patch_arrays, patch_keys = get_patch_arrays(
        patch_year_points, cache_dir=patch_dir, radius=patch_radius,
        download_embeddings=download_embeddings)
    X_patch_all = align_patch_arrays(patch_arrays, patch_keys,
                                     expanded_Glance_Class1[['Glance_ID', 'Year']])
    np.save(patch_all_path, X_patch_all)
    print(f'Saved X_patch_all to {patch_all_path}: {X_patch_all.shape}')

# ── Apply the existing train/test split to the patches ───────────────────────────────
X_train_patch = X_patch_all[x_train.index.values]
X_test_patch = X_patch_all[x_test.index.values]
print(f'Patch windows: {X_patch_all.shape}  (n, k, k, channels)')
print(f'Train/test patches: {X_train_patch.shape} / {X_test_patch.shape}')
print(f'Missing (all-NaN) patches in train: {int(np.isnan(X_train_patch).all(axis=(1, 2, 3)).sum())}')


In [ ]:
# ── CNN (embedding patches) ──────────────────────────────────────────────────────────
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = ""
filename = f'CNN_model{filename_ext}.joblib'

# ── Glance_ID groups (aligned to patch rows; grouped val split for the loss curve) ───
groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_cnn(X_train_patch, y_train, groups=groups_train, verbose=True)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'CNN_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'CNN_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate (no feature importances for a CNN) ──────────────────────────────────────
print(f'Training time: {training_time_sec:.2f} seconds')
cnn_metrics = evaluate_model(
    mdl, X_test_patch, y_test, class_map=class1_dict,
    title='CNN (patches) — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'CNN_confusion{filename_ext}.png'))

# ── Overfitting diagnostic (train vs val loss per epoch) ─────────────────────────────
plot_training_curve(
    getattr(mdl, 'history_', None), title='CNN (patches) — loss curve',
    save_path=os.path.join(MODEL_DIR, f'CNN_losscurve{filename_ext}.png'))


In [ ]:
# ── CNN (early stopping) ─────────────────────────────────────────────────────────────
# Stops when val loss plateaus and restores the best-epoch weights (epochs is the cap).
# ── Variant filename ─────────────────────────────────────────────────────────────────
filename_ext = "_tuned"
filename = f'CNN_model{filename_ext}.joblib'

# ── Glance_ID groups (aligned to patch rows; grouped val split to early-stop on) ─────
groups_train = expanded_Glance_Class1_Clean.loc[x_train.index, "Glance_ID"]

# ── Fit or load ──────────────────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_cnn(X_train_patch, y_train, groups=groups_train,
                                     early_stopping_rounds=15, epochs=200, verbose=True)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'CNN_training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'CNN_training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ─────────────────────────────────────────────────────────────────────────
print(f'Training time: {training_time_sec:.2f} seconds')
best_epoch = getattr(mdl, 'best_epoch_', None)
if best_epoch is not None:
    print(f'Early stopping restored best epoch {best_epoch + 1}')
cnn_tuned_metrics = evaluate_model(
    mdl, X_test_patch, y_test, class_map=class1_dict,
    title='CNN (tuned) — confusion matrix',
    save_path=os.path.join(MODEL_DIR, f'CNN_confusion{filename_ext}.png'))

# ── Overfitting diagnostic (train vs val loss per epoch; stops near best epoch) ──────
plot_training_curve(
    getattr(mdl, 'history_', None), title='CNN (tuned) — loss curve',
    save_path=os.path.join(MODEL_DIR, f'CNN_losscurve{filename_ext}.png'))


## Model Comparison

Load every saved model and compare them on the shared test set, then on the Washington/Oregon subset.

In [ ]:
# ── Compare Models ───────────────────────────────────────────────────────────────────
# Load every saved model + its training time and compare on the shared held-out test set.
def _load_training_time(time_filename):
    with open(os.path.join(MODEL_DIR, time_filename)) as f:
        return float(f.read().split()[0])  # first token of "<seconds> seconds"

# ── Models on the shared single-pixel x_test ─────────────────────────────────────────
model_files = {
    'Random Forest':   ('random_forest_model.joblib',                  'random_forest_training_time.txt'),
    'XGBoost':         ('XGBoost_model.joblib',                        'XGBoost_training_time.txt'),
    'RF (tuned)':      ('random_forest_tuned_model_groupedCV.joblib',  'random_forest_tuned_training_time_groupedCV.txt'),
    'XGBoost (tuned)': ('XGBoost_tuned_model_groupedCV.joblib',        'XGBoost_tuned_training_time_groupedCV.txt'),
    'MLP':             ('MLP_model.joblib',                            'MLP_training_time.txt'),
    'MLP (tuned)':     ('MLP_model_tuned.joblib',                      'MLP_training_time_tuned.txt'),
}
models = {}
for name, (model_file, time_file) in model_files.items():
    models[name] = (joblib.load(os.path.join(MODEL_DIR, model_file)), _load_training_time(time_file))

# ── Models with a different input (pass their own test set as a 3-tuple) ─────────────
cnn_model = joblib.load(os.path.join(MODEL_DIR, 'CNN_model.joblib'))
models['CNN'] = (cnn_model, _load_training_time('CNN_training_time.txt'), X_test_patch)
cnn_tuned_model = joblib.load(os.path.join(MODEL_DIR, 'CNN_model_tuned.joblib'))
models['CNN (tuned)'] = (cnn_tuned_model, _load_training_time('CNN_training_time_tuned.txt'), X_test_patch)
xgb_spatial_model = joblib.load(os.path.join(MODEL_DIR, 'XGBoost_nbhd_tuned_model_spatial.joblib'))
models['XGBoost (spatial)'] = (xgb_spatial_model, _load_training_time('XGBoost_nbhd_tuned_training_time_spatial.txt'), x_test_nbhd)

# ── Compare and display ──────────────────────────────────────────────────────────────
summary_df, per_class_f1_df = compare_models(
    models, x_test, y_test, class_map=class1_dict,
    save_dir=MODEL_DIR, prefix='model_comparison')
print('=== Summary metrics (saved to model_comparison_summary.csv) ===')
display(summary_df.round(4))
print('\n=== Per-class F1 (saved to model_comparison_per_class_f1.csv) ===')
display(per_class_f1_df.round(4))


In [ ]:
# ── Compare Models — Washington/Oregon subset ────────────────────────────────────────
# How many WA/OR observations are in train vs test, and how well do the models predict
# the WA/OR observations within the test set? Reuses the models loaded above.

# ── WA/OR polygons (same TIGER source as the Extra cell) ─────────────────────────────
wa_or_fc = (ee.FeatureCollection('TIGER/2018/States')
            .filter(ee.Filter.inList('NAME', ['Washington', 'Oregon'])))
wa_or_states = gpd.GeoDataFrame.from_features(
    wa_or_fc.getInfo()['features'], crs='EPSG:4326')[['NAME', 'geometry']]

# ── Point-in-polygon: label each Class1 row's state (aligned by index) ───────────────
pts = gpd.GeoDataFrame(
    expanded_Glance_Class1[['Glance_ID']].copy(),
    geometry=gpd.points_from_xy(expanded_Glance_Class1['Lon'], expanded_Glance_Class1['Lat']),
    crs='EPSG:4326')
joined = gpd.sjoin(pts, wa_or_states, predicate='within', how='left')
joined = joined[~joined.index.duplicated(keep='first')]  # guard against border points matching both
state_of = joined['NAME'].reindex(expanded_Glance_Class1.index)
in_wa_or = state_of.notna()

# ── Counts in train vs test ──────────────────────────────────────────────────────────
train_state, test_state = state_of.loc[x_train.index], state_of.loc[x_test.index]
print('WA/OR observations by split:')
print(f"  Training: {int(in_wa_or.loc[x_train.index].sum())} of {len(x_train)} "
      f"(WA={int((train_state == 'Washington').sum())}, OR={int((train_state == 'Oregon').sum())})")
print(f"  Test:     {int(in_wa_or.loc[x_test.index].sum())} of {len(x_test)} "
      f"(WA={int((test_state == 'Washington').sum())}, OR={int((test_state == 'Oregon').sum())})")

# ── Restrict the test set to WA/OR (mask is in x_test / X_test_patch row order) ──────
mask = in_wa_or.loc[x_test.index].values
x_test_wa_or = x_test[mask]
y_test_wa_or = y_test[mask]
X_test_patch_wa_or = X_test_patch[mask]

# ── Re-point the different-input models at the WA/OR subset of their test set ────────
models_wa_or = dict(models)
models_wa_or['CNN'] = (cnn_model, models['CNN'][1], X_test_patch_wa_or)
models_wa_or['CNN (tuned)'] = (cnn_tuned_model, models['CNN (tuned)'][1], X_test_patch_wa_or)
models_wa_or['XGBoost (spatial)'] = (xgb_spatial_model, models['XGBoost (spatial)'][1], x_test_nbhd[mask])

# ── Compare and display ──────────────────────────────────────────────────────────────
print(f'\nEvaluating on {int(mask.sum())} WA/OR test observations...')
summary_wa_or_df, per_class_wa_or_df = compare_models(
    models_wa_or, x_test_wa_or, y_test_wa_or, class_map=class1_dict,
    save_dir=MODEL_DIR, prefix='model_comparison_WA_OR')
print('\n=== WA/OR summary metrics (saved to model_comparison_WA_OR_summary.csv) ===')
display(summary_wa_or_df.round(4))
print('\n=== WA/OR per-class F1 ===')
display(per_class_wa_or_df.round(4))


# Conclusions - Class 1


In [ ]:
#Extra (KEEP AND DON'T EDIT)

# Washington and Oregon boundaries from GEE (TIGER 2018 state boundaries).
states_fc = (ee.FeatureCollection('TIGER/2018/States')
    .filter(ee.Filter.inList('NAME', ['Washington', 'Oregon'])))
states = gpd.GeoDataFrame.from_features(
    states_fc.getInfo()['features'], crs='EPSG:4326'
)[['NAME', 'geometry']]

# glance is already loaded and in EPSG:4326 from the previous cell.
glance_wa_or = gpd.sjoin(glance, states, predicate='within', how='inner')

counts = glance_wa_or['NAME'].value_counts()
for state in ['Washington', 'Oregon']:
    print(f'{state}: {counts.get(state, 0)}')
print(f'Total (WA + OR): {len(glance_wa_or)}')

#glance_wa_or.head()
print(glance.columns)
glance_NA = glance.iloc[glance['Continent_Code']==1,]
print(glance_NA.shape)
print(glance_NA['Glance_Class_ID_level2'].value_counts())

print(glance_NA.shape[0])
Class1 = []
Class2 = []
for i in range(0,glance_NA.shape[0]):
    key = glance_NA['Glance_Class_ID_level1'].iloc[i]
    Class1 = Class1 + [class1_dict[key]]
    key2 = glance_NA['Glance_Class_ID_level2'].iloc[i]
    #print(class2_dict[key2])
    Class2 = Class2 + [class2_dict[key2]]
glance_NA['Class1'] = Class1
glance_NA['Class2'] = Class2
#print(glance_NA.head())
print(glance_NA['Class2'].value_counts())
#print(glance_NA['Glance_Class_ID_level2'].unique())
glance_NA_0 = glance_NA.iloc[glance_NA['Class2']=="Unknown",]
print(glance_NA_0['Class1'].value_counts())

print(glance_NA_filtered['Change'].value_counts())
test = glance_NA_filtered.iloc[glance_NA_filtered['Change'],]
#print(test.head())
test['End_Year'].value_counts()

counts = glance_NA_filtered[['Lat', 'Lon']].value_counts().reset_index(name='count')
print(counts['count'].value_counts())

test2 = glance_NA_filtered.iloc[glance_NA_filtered['Segment_Type']==0,]
print(test2['Change'].value_counts())

print(glance_NA_filtered['Change'].value_counts())
print(glance_NA_filtered.shape)
print(glance_NA_filtered['Class2'].value_counts())